## Useful Websites
- [All ops PDF](https://www.ilovepdf.com/)
- [remove.bg](https://www.remove.bg/)

## PDF to Image

In [ ]:
!pip install -q PyMuPDF

In [ ]:
import fitz  # PyMuPDF
from pathlib import Path

def pdf_page_to_image(pdf_path, output_dir, pages=None, dpi=300):
    """
    Converts specified pages of a PDF to images, maintaining dimensions.

    Args:
        pdf_path (str or Path): The path to the input PDF file.
        output_dir (str or Path): The directory to save the output images.
        pages (list, optional): A list of page numbers (0-indexed) to convert.
                                 If None, all pages are converted.
        dpi (int, optional): The resolution in dots per inch. Higher values result in better quality. Defaults to 300.
    """
    pdf_path = Path(pdf_path)
    output_dir = Path(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    try:
        doc = fitz.open(pdf_path)
        if pages is None:
            pages_to_convert = range(doc.page_count)
        else:
            pages_to_convert = [p for p in pages if 0 <= p < doc.page_count]

        # Set the matrix for higher resolution
        matrix = fitz.Matrix(dpi / 72, dpi / 72)

        for page_num in pages_to_convert:
            page = doc.load_page(page_num)  # Load the specific page
            pix = page.get_pixmap(matrix=matrix)  # Render page to an image (pixmap) with specified resolution

            # Save the pixmap as an image file
            image_path = output_dir / f"page_{page_num + 1}.png"
            pix.save(str(image_path))
            print(f"Converted page {page_num + 1} to {image_path}")

        doc.close()
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage:
# Replace 'input.pdf' with the path to your PDF file
# Replace 'output_images' with the desired output directory
# Replace [0, 2] with the list of page numbers you want to convert (or None for all pages)
# pdf_path = 'input.pdf'
# output_directory = 'output_images'
# pages_to_convert = [0, 1, 2] # Example: convert first three pages
# pdf_page_to_image(pdf_path, output_directory, pages=pages_to_convert, dpi=300)

# To convert all pages with a different DPI:
# pdf_path = 'input.pdf'
# output_directory = 'output_images_high_res'
# pdf_page_to_image(pdf_path, output_directory, dpi=600)

In [ ]:
pdf_path = '/content/2025-09 - MasterCard.pdf'
output_directory = '/content/out'
pdf_page_to_image(pdf_path, output_directory)

In [ ]:
pdf_path = '/content/2025-09 - MasterCard.pdf'
output_directory = '/content/out2'
pages = [10]
pdf_page_to_image(pdf_path, output_directory, pages=[2, 9])

In [ ]:
pdf_path = '/content/2025-09 - MasterCard.pdf'
output_directory = '/content/out_high_res4'
pdf_page_to_image(pdf_path, output_directory, dpi=200)

## Remove image background
   



In [ ]:
!pip install -q rembg onnxruntime

In [ ]:
from rembg import remove
from PIL import Image

def remove_background(input_path, output_path):
    """
    Removes the background from an image.

    Args:
        input_path (str): The path to the input image file.
        output_path (str): The path to save the output image with background removed.
    """
    try:
        # Open the input image
        input_image = Image.open(input_path)

        # Remove the background
        output_image = remove(input_image)

        # Save the output image
        output_image.save(output_path)
        print(f"Background removed and saved to {output_path}")

    except FileNotFoundError:
        print(f"Error: Input file not found at {input_path}")
    except Exception as e:
        print(f"An error occurred: {e}")

# Example usage:
# Replace 'input.png' with the path to your input image file
# Replace 'output_no_bg.png' with the desired output path
# input_image_path = 'input.png'
# output_image_path = 'output_no_bg.png'
# remove_background(input_image_path, output_image_path)

In [ ]:
input_image_path = '/content/sign_new.jpg'
output_image_path = '/content/sign_new_no_bg.png'
remove_background(input_image_path, output_image_path)

## Create PDF from images / list of images

In [ ]:
from PIL import Image
import os

def create_pdf_from_images(image_paths, output_pdf_path):
    """
    Creates a PDF from a list of image paths.

    Args:
        image_paths (list): A list of paths to the input image files.
        output_pdf_path (str): The path to save the output PDF file.
    """
    if not image_paths:
        print("No image paths provided.")
        return

    images = []
    for img_path in image_paths:
        try:
            img = Image.open(img_path)
            # Convert to RGB if not already to avoid issues with saving as PDF
            if img.mode != 'RGB':
                img = img.convert('RGB')
            images.append(img)
        except FileNotFoundError:
            print(f"Error: Image file not found at {img_path}")
        except Exception as e:
            print(f"An error occurred while opening {img_path}: {e}")

    if not images:
        print("No valid images found to create PDF.")
        return

    # Save the first image, and append the rest
    images[0].save(output_pdf_path, save_all=True, append_images=images[1:])
    print(f"PDF created successfully at {output_pdf_path}")

def create_pdf_from_directory(image_dir, output_pdf_path, extensions=['.png', '.jpg', '.jpeg', '.gif']):
    """
    Creates a PDF from all image files in a directory.

    Args:
        image_dir (str): The path to the directory containing image files.
        output_pdf_path (str): The path to save the output PDF file.
        extensions (list, optional): A list of image file extensions to include.
    """
    image_paths = []
    try:
        for filename in os.listdir(image_dir):
            if any(filename.lower().endswith(ext) for ext in extensions):
                image_paths.append(os.path.join(image_dir, filename))
        # Sort image paths to ensure pages are in order (e.g., page_1.png, page_2.png)
        image_paths.sort()
        create_pdf_from_images(image_paths, output_pdf_path)
    except FileNotFoundError:
        print(f"Error: Directory not found at {image_dir}")
    except Exception as e:
        print(f"An error occurred: {e}")


# Example usage from a list of paths:
# image_files = ['/content/out/page_1.png', '/content/out/page_2.png']
# output_pdf = '/content/output_from_list.pdf'
# create_pdf_from_images(image_files, output_pdf)

# Example usage from a directory:
# image_directory = '/content/out'
# output_pdf_from_dir = '/content/output_from_dir.pdf'
# create_pdf_from_directory(image_directory, output_pdf_from_dir)

In [ ]:
img_template = "/content/out_high_res4/page_{0}.png"

image_files = [img_template.format(i) for i in range(1, 11)]
# print(image_files)

output_pdf = '/content/output_from_list.pdf'
create_pdf_from_images(image_files, output_pdf)


## compress pdf size

In [ ]:
!apt-get update
!apt-get install -y ghostscript

In [ ]:
import subprocess
import os

def compress_pdf(input_pdf_path, output_pdf_path, quality='ebook'):
    """
    Compresses a PDF file using Ghostscript.

    Args:
        input_pdf_path (str): The path to the input PDF file.
        output_pdf_path (str): The path to save the compressed output PDF file.
        quality (str): The compression quality preset.
                       Options: 'screen', 'ebook', 'printer', 'prepress', 'default'.
    """
    # Validate quality setting
    valid_qualities = ['screen', 'ebook', 'printer', 'prepress', 'default']
    if quality not in valid_qualities:
        print(f"Error: Invalid quality setting '{quality}'. Choose from: {valid_qualities}")
        return

    # Construct the Ghostscript command
    # -sDEVICE=pdfwrite: Output to PDF
    # -dCompatibilityLevel=1.4: Set PDF compatibility level
    # -dPDFSETTINGS=/{quality}: Use the specified quality setting
    # -dNOPAUSE -dBATCH -q: Standard Ghostscript options for non-interactive batch processing
    # -sOutputFile={output_pdf_path}: Specify the output file
    # {input_pdf_path}: Specify the input file
    command = [
        "gs",
        "-sDEVICE=pdfwrite",
        "-dCompatibilityLevel=1.4",
        f"-dPDFSETTINGS=/{quality}",
        "-dNOPAUSE",
        "-dBATCH",
        "-q",
        f"-sOutputFile={output_pdf_path}",
        input_pdf_path
    ]

    try:
        # Execute the command
        print(f"Compressing {input_pdf_path} to {output_pdf_path} with quality '{quality}'...")
        subprocess.run(command, check=True, capture_output=True, text=True)
        print("Compression complete.")
        # You can optionally check the file size here
        original_size = os.path.getsize(input_pdf_path)
        compressed_size = os.path.getsize(output_pdf_path)
        print(f"Original size: {original_size} bytes")
        print(f"Compressed size: {compressed_size} bytes")

    except FileNotFoundError:
        print("Error: Ghostscript not found. Please ensure it is installed and in your PATH.")
    except subprocess.CalledProcessError as e:
        print(f"Error during Ghostscript execution: {e.stderr}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Example Usage:
# input_pdf = '/content/output_from_list.pdf' # Replace with your input PDF path
# output_pdf_compressed = '/content/output_compressed.pdf' # Replace with your desired output path

# Compress with 'ebook' quality
# compress_pdf(input_pdf, output_pdf_compressed, quality='ebook')

# Compress with 'screen' quality (lower quality, higher compression)
# output_pdf_screen = '/content/output_screen.pdf'
# compress_pdf(input_pdf, output_pdf_screen, quality='screen')

In [ ]:
input_pdf = '/content/output_from_list.pdf' # Replace with your input PDF path
output_pdf_compressed = '/content/output_compressed.pdf' # Replace with your desired output path

# Compress with 'ebook' quality
compress_pdf(input_pdf, output_pdf_compressed, quality='screen')
